# Week 5 Studio — Diffie–Hellman, MITM & the Trust Anchor

**Task brief:** [`README.md`](README.md) · **Session A notebook:** [`../../notebooks/week-05-protocols-tls-pki.ipynb`](../../notebooks/week-05-protocols-tls-pki.ipynb) · **Given engine:** [`dh_pki.py`](dh_pki.py) · **Rubric:** [`../../resources/control-scorecard.md`](../../resources/control-scorecard.md)

This notebook completes all four studio tasks:

| Task | What | Where |
|---|---|---|
| 1 | DH key agreement + the man-in-the-middle | §1 |
| 2 | Certificate-chain validation & forgery rejection | §2 |
| 3 | The trust-store attack (poisoning) | §3 |
| — | The provided test suite, run unmodified | §4 |
| — | Bypass attempts beyond the tests (scorecard axis 4) | §5 |
| 4 | Control Scorecard for DH, cert chains and TLS 1.3 | §6 |

The primitives (`DH_P`, `DH_G`, toy RSA, `make_cert`, `validate`) come from the given
`dh_pki.py` and are **not** reimplemented. The four functions from `starter.py` are
implemented here and then plugged into `test_dh_pki.py` unchanged.

> **Warm-up — *DH gave Alice and Bob a shared secret over a wire the attacker controls. Why isn't that enough, and what fills the gap?***
> DH only guarantees that *whoever is on the other end* shares the secret with you.
> It says nothing about *who* that is. An active attacker who can rewrite the wire
> simply becomes the other end, twice. What fills the gap is **authentication**:
> binding the DH public value to an identity with a signature, and binding that
> signing key to a name via a **certificate chain** that ends in a trusted root (PKI).

In [1]:
# --- bootstrap: works from the repo root or from studios/week-05 ---
import sys, pathlib, types, random, time, os
here = pathlib.Path.cwd()
week = here if (here / "dh_pki.py").exists() else here / "studios" / "week-05"
if str(week) not in sys.path:
    sys.path.insert(0, str(week))

import dh_pki
from dh_pki import DH_P, DH_G, make_cert, validate, verify, sign, gen_key, load_fixtures

fx = load_fixtures()
print(f"DH group: p is {DH_P.bit_length()} bits, g = {DH_G}")
print("fixture p matches dh_pki.DH_P:", fx["dh"] == (DH_P, DH_G))
print("certs in the legitimate chain:", [c["subject"] for c in fx["chain"]])
print("trust store (root public keys):", len(fx["trust_store"]))

DH group: p is 768 bits, g = 2
fixture p matches dh_pki.DH_P: True
certs in the legitimate chain: ['bank.example.com', 'ACME Intermediate', 'ACME Root CA']
trust store (root public keys): 1


## 1 · Task 1 — Diffie–Hellman and the man-in-the-middle

> **Note on the group.** `dh_pki.py` and the README call `DH_P` "the RFC 2409 group-2
> 1024-bit prime", but the constant is **768 bits** (the bootstrap cell prints
> `p.bit_length()`). It is RFC 2409 **group 1**, whose hex is the first 192 digits of
> group 2's. This doesn't affect any task, but it matters for the scorecard: 768-bit
> discrete logs have been computed publicly (2016).

Alice and Bob each pick a private exponent, publish `g^x mod p`, and raise the other
side's public value to their own private exponent. Both land on `g^(ab) mod p`.
A **passive** eavesdropper sees only `g^a` and `g^b`; computing `g^(ab)` from those is
the Diffie–Hellman problem, believed hard in this group (discrete-log assumption).

Mallory is **active**: she drops Alice's and Bob's public values and injects her own
`M = g^m` toward *both* sides. Each honest party then completes a perfectly correct
DH exchange — with Mallory.

In [2]:
def dh_public(private, g=DH_G, p=DH_P):
    '''Alice/Bob's public value: g^private mod p, sent over the wire.'''
    return pow(g, private, p)


def dh_shared(their_public, my_private, p=DH_P):
    '''The shared secret each side computes: their_public^my_private mod p.'''
    return pow(their_public, my_private, p)


def mitm_keys(a, b, m, g=DH_G, p=DH_P):
    '''Mallory (private m) injects her public value toward BOTH Alice and Bob.'''
    A = dh_public(a, g, p)          # Alice -> (intercepted)
    B = dh_public(b, g, p)          # Bob   -> (intercepted)
    M = dh_public(m, g, p)          # Mallory -> Alice AND Mallory -> Bob

    alice = dh_shared(M, a, p)          # Alice thinks M came from Bob
    mallory_alice = dh_shared(A, m, p)  # Mallory's key for the Alice leg
    bob = dh_shared(M, b, p)            # Bob thinks M came from Alice
    mallory_bob = dh_shared(B, m, p)    # Mallory's key for the Bob leg

    return {
        "alice": alice,
        "mallory_alice": mallory_alice,
        "bob": bob,
        "mallory_bob": mallory_bob,
        "alice_equals_bob": alice == bob,
    }

### Honest DH — both sides agree on `g^(ab) mod p`

Same fixed private exponents as `test_dh_pki.py`, so the run is deterministic.

In [3]:
A_PRIV = 0x1a2b3c4d5e6f7788
B_PRIV = 0x99aabbccddeeff00
M_PRIV = 0xdeadbeefcafef00d      # Mallory

A, B = dh_public(A_PRIV), dh_public(B_PRIV)
k_alice = dh_shared(B, A_PRIV)
k_bob   = dh_shared(A, B_PRIV)

print("Alice's key == Bob's key :", k_alice == k_bob)
print("key == g^(ab) mod p      :", k_alice == pow(DH_G, A_PRIV * B_PRIV, DH_P))
print("what the eavesdropper sees: A =", hex(A)[:34] + "...", " B =", hex(B)[:34] + "...")
print("shared secret (prefix)   :", hex(k_alice)[:34] + "...")

Alice's key == Bob's key : True
key == g^(ab) mod p      : True
what the eavesdropper sees: A = 0x31777eb497d24fc909a4172bd87d496e...  B = 0x61ba3b2605b75ff8ba5cf7893f819017...
shared secret (prefix)   : 0xaf4d0c201032fc3cfa4ace4ca9cba0ad...


### ⚠️ The MITM — the guarantee collapses

This is the `test_mitm_breaks_unauthenticated_dh` guarantee test, run inline.

In [4]:
mk = mitm_keys(A_PRIV, B_PRIV, M_PRIV)

print("Alice   <-> Mallory share a key:", mk["alice"] == mk["mallory_alice"])
print("Bob     <-> Mallory share a key:", mk["bob"] == mk["mallory_bob"])
print("Alice   <-> Bob     share a key:", mk["alice_equals_bob"], "  <- the guarantee failing")

assert mk["alice"] == mk["mallory_alice"] and mk["bob"] == mk["mallory_bob"]
assert mk["alice_equals_bob"] is False

# Mallory's relay in miniature: decrypt with one leg's key, re-encrypt with the other.
import hashlib
def toy_encrypt(key_int, msg):
    ks = hashlib.sha256(key_int.to_bytes(128, "big")).digest()
    return bytes(x ^ y for x, y in zip(msg, ks))

c_from_alice = toy_encrypt(mk["alice"], b"transfer $100 to Bob")
read_by_mallory = toy_encrypt(mk["mallory_alice"], c_from_alice)
tampered = read_by_mallory.replace(b"$100", b"$999").replace(b"Bob", b"Mal")
c_to_bob = toy_encrypt(mk["mallory_bob"], tampered)
print()
print("Mallory reads  :", read_by_mallory)
print("Bob receives   :", toy_encrypt(mk["bob"], c_to_bob), "(decrypts cleanly -- no alarm)")

Alice   <-> Mallory share a key: True
Bob     <-> Mallory share a key: True
Alice   <-> Bob     share a key: False   <- the guarantee failing

Mallory reads  : b'transfer $100 to Bob'
Bob receives   : b'transfer $999 to Mal' (decrypts cleanly -- no alarm)


**What DH guarantees, as a conditional (axis 2):** DH gives Alice and Bob a shared
secret that a **passive** eavesdropper cannot compute (under the discrete-log /
CDH assumption), **provided the endpoints are authenticated**, i.e. each side knows the
public value it received really came from the other party.

Remove the proviso and the MITM walks straight in: every leg is still perfectly secret,
but it is secret *with Mallory*. Both legs decrypt cleanly, so nothing alerts anyone.
**Secrecy without authentication is a private conversation with an impostor.**

**Fast-finisher: where does the "authenticated" in TLS 1.3's authenticated DH come from?**
The server signs the handshake transcript, which includes both DH key shares, in the
`CertificateVerify` message. It signs with the private key whose public half is bound to
its hostname by the certificate chain from §2. A substituted key share breaks that
signature, and a substituted certificate breaks chain validation.

The cell below demonstrates that fix with the week's own primitives. Bob (the bank)
signs his DH public value with the key bound to `bank.example.com`. To do that we need
the bank's private key, so we regenerate the notebook's PKI from its RNG seed, playing
the role of the legitimate key owners.

In [5]:
# Regenerate the notebook's key material (same seed and draw order as dh_pki.build_pki).
rng = random.Random(7)
root_pub,  root_priv  = gen_key(rng)
inter_pub, inter_priv = gen_key(rng)
leaf_pub,  leaf_priv  = gen_key(rng)          # bank.example.com's key
mal_pub,   mal_priv   = gen_key(rng)          # Mallory's key
rogue_pub, rogue_priv = gen_key(rng)          # Mallory's rogue CA
assert leaf_pub == fx["chain"][0]["pub"] and root_pub in fx["trust_store"]

def share_msg(dh_value):
    return f"DH-share|{dh_value}".encode()

def alice_accepts(dh_value, sig, chain, trust_store, expected_host="bank.example.com"):
    '''Alice's side of authenticated DH: chain valid, right name, key share signed by that key.'''
    ok, reason = validate(chain, trust_store)
    if not ok:
        return False, reason
    if chain[0]["subject"] != expected_host:
        return False, f"certificate is for {chain[0]['subject']!r}, not {expected_host!r}"
    if not verify(chain[0]["pub"], share_msg(dh_value), sig):
        return False, "key share not signed by the certified key -> MITM detected"
    return True, "authenticated key share"

# Honest run: Bob sends B + signature + his chain.
print("honest   :", alice_accepts(B, sign(leaf_priv, share_msg(B)), fx["chain"], fx["trust_store"]))

# Mallory swaps in M but can only replay Bob's signature over B...
M = dh_public(M_PRIV)
print("MITM #1  :", alice_accepts(M, sign(leaf_priv, share_msg(B)), fx["chain"], fx["trust_store"]))
# ...or sign M with her own key, which is not the certified one...
print("MITM #2  :", alice_accepts(M, sign(mal_priv, share_msg(M)), fx["chain"], fx["trust_store"]))
# ...or also bring her own chain, which ends in an untrusted root.
print("MITM #3  :", alice_accepts(M, sign(mal_priv, share_msg(M)),
                                  [fx["rogue_leaf"], fx["rogue_ca"]], fx["trust_store"]))

honest   : (True, 'authenticated key share')
MITM #1  : (False, 'key share not signed by the certified key -> MITM detected')
MITM #2  : (False, 'key share not signed by the certified key -> MITM detected')
MITM #3  : (False, "anchor 'Rogue CA' is not a trusted root")


Once the key share is signed, all three MITM variants are rejected. Each rejection lands
on a *different* check: the share signature, the share signature again, and the trust
anchor. Authentication moved the question from "is the math right?" to "do I trust the
key that signed this?", which is exactly what Tasks 2 and 3 examine.

## 2 · Task 2 — Certificate-chain validation and forgery rejection

`validate(chain, trust_store)` (given) walks leaf → intermediate → root. It checks that each
cert is signed by the next one, then requires the last cert's key to be **in the trust
store** and its self-signature to verify. We run it on the class chain and the two forgeries.

In [6]:
cases = {
    "legitimate chain (leaf <- inter <- root)": fx["chain"],
    "forgery 1: self-signed bank cert        ": [fx["self_signed_forgery"]],
    "forgery 2: rogue-CA chain               ": [fx["rogue_leaf"], fx["rogue_ca"]],
}
for name, chain in cases.items():
    ok, reason = validate(chain, fx["trust_store"])
    print(f"{name} -> {ok!s:5}  {reason}")

# The forgeries are internally CONSISTENT: every signature in them is valid.
print()
print("self-signed forgery's own signature verifies:",
      verify(fx["self_signed_forgery"]["pub"], fx["self_signed_forgery"]["body"], fx["self_signed_forgery"]["sig"]))
print("rogue leaf verifies under rogue CA           :",
      verify(fx["rogue_ca"]["pub"], fx["rogue_leaf"]["body"], fx["rogue_leaf"]["sig"]))

legitimate chain (leaf <- inter <- root) -> True   chain valid — key is authentically bound to the name
forgery 1: self-signed bank cert         -> False  anchor 'bank.example.com' is not a trusted root
forgery 2: rogue-CA chain                -> False  anchor 'Rogue CA' is not a trusted root

self-signed forgery's own signature verifies: True
rogue leaf verifies under rogue CA           : True


**Why both forgeries fail at the same check:** both chains are cryptographically
well-formed; every signature in them verifies. They fail only because they end in a key
that is **not in the trust store**. Mallory can sign anything, but she cannot make your
browser trust her signing key. That is what PKI buys, and it reduces the whole problem
to one question: **is the trust store correct?**

### Coverage over a forgery corpus (axis 3)

Two forgeries is a sample of two. Below is a corpus of 26 distinct forgery attempts
under the clean trust store. Each is a variant Mallory can build **without** any key the
store trusts: fresh self-signed certs, fresh rogue CAs, rogue intermediates, a real root
appended to a rogue leaf, and a real leaf with a tampered subject or key.

In [7]:
def forgery_corpus(n_keys=6, seed=2025):
    r = random.Random(seed)
    root_cert, inter_cert, leaf_cert = fx["chain"][2], fx["chain"][1], fx["chain"][0]
    corpus = []
    for i in range(n_keys):
        m_pub, m_priv = gen_key(r)
        c_pub, c_priv = gen_key(r)
        rca = make_cert(f"Rogue CA {i}", c_pub, c_priv)
        corpus += [
            (f"self-signed #{i}",             [make_cert("bank.example.com", m_pub, m_priv)]),
            (f"rogue CA #{i}",                [make_cert("bank.example.com", m_pub, c_priv), rca]),
            (f"rogue leaf + real root #{i}",  [make_cert("bank.example.com", m_pub, c_priv), root_cert]),
            (f"rogue leaf + real inter #{i}", [make_cert("bank.example.com", m_pub, c_priv), inter_cert, root_cert]),
        ]
    # Tampering with the legitimate leaf: swap the key or the name, keep the old signature.
    swapped_key = dict(leaf_cert, pub=mal_pub, body=f"bank.example.com|{mal_pub}".encode())
    swapped_name = dict(leaf_cert, subject="evil.example.com",
                        body=f"evil.example.com|{leaf_cert['pub']}".encode())
    corpus += [("real leaf, key swapped", [swapped_key, inter_cert, root_cert]),
                   ("real leaf, name swapped", [swapped_name, inter_cert, root_cert])]
    return corpus

corpus = forgery_corpus()
blocked, by_reason = 0, {}
for name, chain in corpus:
    ok, reason = validate(chain, fx["trust_store"])
    blocked += not ok
    key = "untrusted anchor" if "trusted root" in reason else "bad signature in chain"
    by_reason[key] = by_reason.get(key, 0) + (not ok)

print(f"coverage: {blocked}/{len(corpus)} forgeries blocked under the clean trust store")
for k, v in by_reason.items():
    print(f"  {v:2} rejected with: {k}")

coverage: 26/26 forgeries blocked under the clean trust store
  12 rejected with: untrusted anchor
  14 rejected with: bad signature in chain


### False-positive cost (axis 5)

A validator that rejects everything would score 26/26 as well. So we also need a
**benign corpus**: 20 legitimate certificates issued by the real ACME intermediate for
different hostnames (we act as the CA, using the regenerated issuing key). Each must
validate.

In [8]:
root_cert, inter_cert = fx["chain"][2], fx["chain"][1]
r = random.Random(99)
benign = []
for i in range(20):
    host_pub, _ = gen_key(r)
    benign.append([make_cert(f"site{i}.example.com", host_pub, inter_priv), inter_cert, root_cert])

rejected = sum(not validate(ch, fx["trust_store"])[0] for ch in benign)
print(f"false positives: {rejected}/{len(benign)} legitimate chains rejected")

false positives: 0/20 legitimate chains rejected


## 3 · Task 3 — The trust-store attack

The validator is sound, so Mallory attacks its **input** instead: the trust store.
Malware, a coerced admin, or a "corporate inspection" installer adds her root. We build
a **new** store, leaving the caller's untouched, so the before/after comparison uses
the same original store.

In [9]:
def poison_trust_store(trust_store, rogue_root):
    '''Return a NEW trust store that also trusts rogue_root's public key.'''
    return set(trust_store) | {rogue_root["pub"]}

In [10]:
rogue_chain = [fx["rogue_leaf"], fx["rogue_ca"]]
clean = fx["trust_store"]

print("rogue chain, clean store    :", validate(rogue_chain, clean))
poisoned = poison_trust_store(clean, fx["rogue_ca"])
print("rogue chain, poisoned store :", validate(rogue_chain, poisoned))
print()
print("caller's store untouched    :", fx["rogue_ca"]["pub"] not in clean, f"(clean has {len(clean)} root, poisoned has {len(poisoned)})")
print("legit chain still valid too :", validate(fx["chain"], poisoned)[0], " <- nothing looks wrong to the user")

rogue chain, clean store    : (False, "anchor 'Rogue CA' is not a trusted root")
rogue chain, poisoned store : (True, 'chain valid — key is authentically bound to the name')

caller's store untouched    : True (clean has 1 root, poisoned has 2)
legit chain still valid too : True  <- nothing looks wrong to the user


Same chain, same signatures, same validator. The only change is one key added to a set,
and the verdict flips from **rejected** to **accepted**. The signatures never broke; the
trust anchor did. Worse, the poisoned store still accepts every legitimate site, so the
user sees no errors, nothing gets logged and no alert fires. Mallory can now pair this
with §1's MITM: she presents `rogue_leaf` and signs her own DH share, and `alice_accepts`
passes:

In [11]:
M = dh_public(M_PRIV)
print("authenticated DH, clean store   :",
      alice_accepts(M, sign(mal_priv, share_msg(M)), rogue_chain, clean))
print("authenticated DH, poisoned store:",
      alice_accepts(M, sign(mal_priv, share_msg(M)), rogue_chain, poisoned))

authenticated DH, clean store   : (False, "anchor 'Rogue CA' is not a trusted root")
authenticated DH, poisoned store: (True, 'authenticated key share')


### A second route to the same failure: the CA's key is compromised

Poisoning needs write access to the victim's store. The **DigiNotar** route does not:
compromise a CA that is *already* trusted and issue a genuine-looking cert. Our toy PKI
has exactly this flaw. Its keys come from `random.Random(7)`, a public seed. That is a
week-4 entropy failure: anyone can regenerate the root's private key, as we did in §1.
The cell below shows Mallory getting her own key accepted as `bank.example.com` under
the **clean** trust store.

In [12]:
# Mallory, knowing the seed, re-derives the ACME Root's private key and signs directly.
mis_issued = make_cert("bank.example.com", mal_pub, root_priv)
print("mis-issued chain, CLEAN store:", validate([mis_issued, fx["chain"][2]], clean))
print("key Mallory is now 'the bank' with == Mallory's key:", mis_issued["pub"] == mal_pub)

mis-issued chain, CLEAN store: (True, 'chain valid — key is authentically bound to the name')
key Mallory is now 'the bank' with == Mallory's key: True


**Real-world mapping of Task 3:**

| Toy model | Real incident / class |
|---|---|
| `poison_trust_store(...)` | **Malware or a local admin installs a rogue root.** Superfish adware (Lenovo, 2015) shipped a root whose private key was the same on every machine and was trivially extracted. Corporate TLS-inspection roots are the "legitimate" version. |
| Root key re-derived from the public seed | **CA compromised or coerced.** DigiNotar (2011): attackers issued a valid `*.google.com` cert, used to MITM Gmail for ~300 000 users in Iran. Every browser accepted it because DigiNotar was in every trust store. |
| A trusted CA signing for the wrong name | **Negligent mis-issuance.** Symantec test certs for `google.com` (2015), which led to Symantec's roots being distrusted. |

**Certificate Transparency** doesn't *prevent* any of these. It requires every publicly
trusted cert to be logged in append-only public logs (browsers require SCTs), so a
mis-issued `bank.example.com` cert becomes **visible** and the domain owner can detect it.
That is a *detective* control, which foreshadows week 7: when you can't prevent, you
instrument. Note that CT does *not* help against a locally installed rogue root, because
browsers skip CT enforcement for user-installed roots.

## 4 · The provided test suite, run unmodified

`test_dh_pki.py` does `import starter as s`. We register the functions above as the
`starter` module, so the **unmodified** tests grade exactly the code in this notebook,
including both guarantee tests.

In [13]:
starter_mod = types.ModuleType("starter")
for fn in (dh_public, dh_shared, mitm_keys, poison_trust_store):
    setattr(starter_mod, fn.__name__, fn)
sys.modules["starter"] = starter_mod

sys.modules.pop("test_dh_pki", None)
import test_dh_pki
test_dh_pki.main()

  ok  honest DH: Alice and Bob agree on g^(ab) mod p
  ok  MITM: Mallory holds a key with each side; Alice<->Bob share nothing
  ok  legitimate leaf<-intermediate<-root chain validates
  ok  self-signed forgery rejected at the trust anchor
  ok  rogue-CA chain rejected at the SAME check (untrusted anchor)
  ok  trust-store poisoning: rejected clean, ACCEPTED once rogue root trusted

all 6 tests pass


## 5 · Bypass attempts beyond the tests (axis 4)

> *A defense you have not tried to bypass is a defense you have not evaluated.*

The Control Scorecard awards zero on axis 4 without an attempt. Besides the two bypasses
already shown (trust-store poisoning, CA-key compromise), here are two more: one against
DH and one against the chain validator.

### 5a · DH: invalid public values (no private key needed)

Mallory doesn't even need `m`. If she injects `M = 1` or `M = p − 1`, the victim's
"secret" `M^a mod p` can only be `1` or `p − 1`, whatever `a` is. The given group is a
safe prime (`p = 2q + 1`), so the fix is a public-key check: require `1 < Y < p − 1` and
`Y^q ≡ 1 (mod p)` (NIST SP 800-56A).

In [14]:
Q = (DH_P - 1) // 2
def dh_public_ok(Y, p=DH_P):
    return 1 < Y < p - 1 and pow(Y, Q, p) == 1

for label, bad in [("M = 1", 1), ("M = p-1", DH_P - 1), ("M = 0", 0)]:
    secrets = {dh_shared(bad, a) for a in (A_PRIV, B_PRIV, 12345, 67890)}
    print(f"{label:8} -> victim's 'secret' takes {len(secrets)} value(s) {sorted((str(s) if s < 10 else 'p-1' for s in secrets))}; "
          f"validated? {dh_public_ok(bad)}")
print("honest A  -> validated?", dh_public_ok(dh_public(A_PRIV)))

M = 1    -> victim's 'secret' takes 1 value(s) ['1']; validated? False
M = p-1  -> victim's 'secret' takes 2 value(s) ['1', 'p-1']; validated? False
M = 0    -> victim's 'secret' takes 1 value(s) ['0']; validated? False
honest A  -> validated? True


Result: **working bypass** of naïve DH, closed by a cheap check. Note that the `starter.py`
API (like the notebook's) performs no such validation. TLS 1.3 requires it for finite-field
groups (RFC 8446 §4.2.8.1).

### 5b · Chain validator: no hostname check

`validate(chain, trust_store)` never sees the hostname the user asked for. The result
says the key is bound to *a* name, not to *the* name. So a cert that ACME issued
**correctly** to `mallory.example.com` validates when Alice asked for `bank.example.com`.
No forgery and no compromise are needed; Mallory just buys a real cert for her own domain.

In [15]:
mallory_real_cert = make_cert("mallory.example.com", mal_pub, inter_priv)   # legitimately issued
chain_for_mallory = [mallory_real_cert, fx["chain"][1], fx["chain"][2]]

print("validate() (what the given engine checks):", validate(chain_for_mallory, clean))

def validate_for_host(chain, trust_store, hostname):
    ok, reason = validate(chain, trust_store)
    if ok and chain[0]["subject"] != hostname:
        return False, f"name mismatch: cert is for {chain[0]['subject']!r}, wanted {hostname!r}"
    return ok, reason

print("validate_for_host(..., 'bank.example.com'):", validate_for_host(chain_for_mallory, clean, "bank.example.com"))
print("validate_for_host(legit, 'bank.example.com'):", validate_for_host(fx["chain"], clean, "bank.example.com"))

validate() (what the given engine checks): (True, 'chain valid — key is authentically bound to the name')
validate_for_host(..., 'bank.example.com'): (False, "name mismatch: cert is for 'mallory.example.com', wanted 'bank.example.com'")
validate_for_host(legit, 'bank.example.com'): (True, 'chain valid — key is authentically bound to the name')


Result: **working bypass** of the given validator, closed by a hostname check. Real stacks
have shipped this exact bug (Georgiev et al., *"The Most Dangerous Code in the World"*,
2012). The toy validator also skips expiry, key-usage / `basicConstraints` (is the
intermediate *allowed* to act as a CA?) and revocation, which are all conditions a real
validator adds. Python's own TLS defaults turn hostname checking on:

In [16]:
import ssl
ctx = ssl.create_default_context()          # local only -- no network connection is made
print("check_hostname      :", ctx.check_hostname)
print("verify_mode         :", ctx.verify_mode.name)
print("TLS 1.3 available   :", ssl.HAS_TLSv1_3)
print("OpenSSL             :", ssl.OPENSSL_VERSION)
roots = ctx.get_ca_certs()
print(f"trusted roots loaded: {len(roots)}  <- each one can vouch for ANY hostname (the real attack surface)")

check_hostname      : True
verify_mode         : CERT_REQUIRED
TLS 1.3 available   : True
OpenSSL             : OpenSSL 3.0.13 30 Jan 2024
trusted roots loaded: 69  <- each one can vouch for ANY hostname (the real attack surface)


### Operational cost (axis 6)

In [17]:
def bench(fn, n):
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1e3

a256 = int.from_bytes(os.urandom(32), "big")
print(f"dh_public ({DH_P.bit_length()}-bit modexp, 256-bit exponent) : {bench(lambda: dh_public(a256), 200):.3f} ms")
print(f"dh_shared                                    : {bench(lambda: dh_shared(B, a256), 200):.3f} ms")
print(f"validate  (3-cert chain, toy 128-bit RSA)    : {bench(lambda: validate(fx['chain'], clean), 2000):.3f} ms")

dh_public (768-bit modexp, 256-bit exponent) : 0.683 ms


dh_shared                                    : 0.894 ms


validate  (3-cert chain, toy 128-bit RSA)    : 0.029 ms


## 6 · Task 4 — Control Scorecard

### The conditional form (axis 2): guarantee + condition

| Mechanism | Guarantee (axis 2) | Its condition / failure | Shown in |
|---|---|---|---|
| **Diffie–Hellman** | Shared secret that a **passive** eavesdropper cannot compute (CDH / discrete log hard in the group; requires ≥2048 bits today, and the studio's is 768) | **provided** both endpoints are authenticated and received public values are validated. With **no authentication**, a MITM holds a key with each side (§1). Unvalidated `Y ∈ {0, 1, p−1}` forces a known secret (§5a). | §1, §5a, `test_mitm_breaks_unauthenticated_dh` |
| **Certificate chain** | Binds a public key to a **name**: only a holder of a trusted CA key could have produced this binding | **provided** the **root trust store** is correct, every CA in it keeps its key secret and issues correctly, **and** the verifier checks the name it actually wanted. A poisoned store (§3), a compromised or mis-issuing CA (§3) or a missing hostname check (§5b) each defeat it. | §2, §3, §5b, `test_trust_store_poisoning_accepts_forgery` |
| **TLS 1.3** | Confidential, integrity-protected, **server-authenticated** channel with forward secrecy | **provided every underlying condition holds**: authenticated (EC)DHE via `CertificateVerify` over the transcript (§1 fix), a correct trust store + hostname check (§2–3, §5b), AEAD nonces never reused under one key (wk 3), strong keys with good entropy (wk 4). | whole notebook |

TLS 1.3 = **authenticated DH** + a **certificate chain** + **AEAD** (AES-GCM, week 3).
Every flaw from the crypto weeks lives somewhere in that stack: nonce reuse in the AEAD
(wk 3), a weak or predictable key (wk 4, and the seed-7 root in §3), a MITM on
unauthenticated DH (§1), a bad trust anchor (§3).

### Full scorecard — Diffie–Hellman (unauthenticated, as in `starter.py`)

| Axis | Before (no key exchange) | After control | Evidence |
|---|---|---|---|
| Threat model | passive eavesdropper **and** active on-path attacker (can drop / inject messages), no credentials | unchanged | §1 |
| Guarantee | none | shared secret vs. a **passive** eavesdropper **provided** endpoints are authenticated and public values validated | §1 |
| Coverage | passive attacker reads everything | passive attacker: 0 secrets learned. Active attacker: **1/1 MITM runs succeeded** (deterministic, so repeating it adds nothing) | §1, test suite |
| **Bypass** | — | **found (×2):** (1) MITM by injecting `M` both ways; (2) invalid public value `M ∈ {0, 1, p−1}` forces a known secret, no private key needed | §1, §5a |
| FP cost | — | 0 (honest parties always agree) | §1 honest run |
| Op cost | — | ≈ 1 modexp per side per value, milliseconds (§5 bench); the given prime is only **768 bits** (group 1, not the 1024-bit group 2 the README states), far below today's 2048-bit minimum | §5 bench |
| Observability | — | **none**: a MITM'd exchange is indistinguishable from an honest one; both legs decrypt cleanly | §1 relay demo |
| Failure mode | — | **fails silently**: no error or log when a MITM is present | §1 |

### Full scorecard — Certificate-chain validation (`validate`)

| Axis | Before (no PKI) | After control | Evidence |
|---|---|---|---|
| Threat model | active on-path attacker who can generate keys and certs, **no** trusted CA key | unchanged. Extended variants: attacker with write access to the trust store, or holding a CA key | §2, §3 |
| Guarantee | none (any key could claim any name) | key bound to a name **provided** the trust store is correct, CA keys are secret and the verifier checks the hostname | §2 |
| Coverage | 0/26 forgeries blocked | **26/26** forgeries blocked under the clean store (untrusted anchor or bad signature) | §2 corpus |
| **Bypass** | — | **found (×3):** trust-store poisoning flips a rejected chain to accepted; CA private key re-derived from public seed 7 (DigiNotar analogue); no hostname check accepts a real cert for `mallory.example.com` | §3, §5b |
| FP cost | — | **0/20** legitimate chains rejected (benign corpus) | §2 benign corpus |
| Op cost | — | sub-millisecond per 3-cert chain (toy RSA); store maintenance and CA fees in reality | §5 bench |
| Observability | — | rejections return a reason string. **Acceptance of a poisoned or mis-issued chain produces nothing.** In reality, CT logs make mis-issuance *detectable* after the fact | §2, §3 |
| Failure mode | — | **fails closed** on a bad chain; **fails open and silent** when the trust anchor itself is wrong | §3 |

### Full scorecard — TLS 1.3 (composition)

| Axis | Before (plaintext) | After control | Evidence |
|---|---|---|---|
| Threat model | on-path active attacker (Wi-Fi, ISP, BGP hijack) | unchanged | — |
| Guarantee | none | confidential + authenticated channel with forward secrecy **provided** all of: authenticated (EC)DHE, correct trust store + hostname check, unique AEAD nonces, well-generated keys | §6 table |
| Coverage | — | each composed piece tested above: MITM blocked once the share is signed (3/3 variants, §1); forgeries 26/26 (§2). Real TLS not measured here | §1, §2 |
| **Bypass** | — | not a break of the TLS math; any single broken **condition** suffices: rogue root (§3), mis-issued cert (§3), a client that skips hostname checks (§5b), nonce reuse (wk 3), weak RNG keys (wk 4) | §3, §5 |
| FP cost | — | legitimate sites fail if they misconfigure chains or expire; corporate inspection breaks pinned apps (not measured here) | — |
| Op cost | — | 1-RTT handshake, one ECDHE + one signature per connection; cert lifecycle management | §5 bench (DH part) |
| Observability | — | handshake failures are logged by clients and servers; CT makes public mis-issuance detectable; locally installed roots are **not** in CT | §3 notes |
| Failure mode | — | **fails closed** on bad cert / bad signature; **fails open and silent** under a poisoned store or mis-issued cert | §3 |

### Where we may have been unfair, and what we did not test

- **Toy parameters.** 64-bit-prime RSA (128-bit moduli) is factorable in minutes with a
  real tool. We didn't factor, because §3 recovered the root key from its public seed,
  which is cheaper. Our results say nothing about the strength of real 2048-bit RSA or
  ECDSA, and the DH prime is 768 bits, a size whose discrete logs are already publicly computable (and TLS 1.3 only allows ≥2048-bit FFDHE groups).
- **We played the CA.** The benign corpus and the authenticated-DH demo use keys
  regenerated from seed 7. A real attacker only has that ability because of the seed
  weakness, which we list as a bypass rather than hide.
- **Deterministic controls.** Coverage numbers are over *our* corpora (26 forgeries,
  20 benign chains). Every forgery variant is structurally one of four shapes, so the
  corpus is a sample of the shapes we thought of, not the whole attack space.
- **Not tested:** expiry, revocation (OCSP/CRL), `basicConstraints` / path length, name
  constraints, wildcard matching, CT/SCT enforcement, downgrade to TLS 1.2, a real TLS
  handshake against a live server. `alice_accepts` signs only the key share, not the full
  transcript as TLS 1.3 does, so it would not catch replay of an old signed share.
- **Axis 5/6 for TLS** are argued from knowledge, not measured.

## Debrief

- **Secrecy and authentication are different properties**, and a channel needs both.
  DH gave perfect secrecy on each leg and zero authentication (§1).
- ⚠️ **Unauthenticated DH → MITM.** Authentication comes from signing the key exchange
  with a key that PKI binds to a name (§1 fix, TLS 1.3 `CertificateVerify`).
- **PKI's guarantee is only as strong as the trust store.** Both forgeries failed at the
  *same* check (untrusted anchor). Poisoning the store, or compromising a trusted CA,
  flipped the verdict without breaking one signature (§3). CT is the *detective* control.
- **TLS 1.3 composes it all.** Same lesson as every crypto week: **name the condition,
  find the attack.**